# Machine Learning for Graduate Placement Classification

This project trains a machine learning model to classify whether a graduate student will receive a job placement offer, using the Campus Recruitment dataset sourced from Kaggle (215 records, 13 features after preprocessing). The dataset contains academic performance scores across secondary, higher secondary, undergraduate and MBA levels, alongside categorical features including gender, degree stream, work experience, and MBA specialisation. The target variable is binary: Placed or Not Placed. This framing is relevant to consulting recruitment contexts, where placement outcome models are increasingly used to filter candidates at scale.

## Procedure

Preprocessing removed `sl_no` (arbitrary index) and `salary` (target leakage — only populated for placed students). Categorical features were label-encoded and numeric features standardised using `StandardScaler`, with all transformers fit on training data only to prevent leakage. The dataset was split 80/20 into training (172 rows) and test (43 rows) sets using stratified sampling to preserve the 68.8/31.2 class ratio. Weighted F1-score was selected as the primary evaluation metric given the moderate 2.2:1 class imbalance, as raw accuracy would be inflated by the majority class.

Three classifiers were trained: Logistic Regression, Decision Tree (max depth 4), and Random Forest (200 estimators), all with `class_weight='balanced'`. Five-fold stratified cross-validation confirmed whether test results were stable across different splits.

## Model metrics & performance on unseen data

| Model | Test accuracy | Test F1 | Train F1 | Overfit gap | CV F1 (mean ± std) |
|---|---|---|---|---|---|
| Logistic Regression | 0.860 | 0.865 | 0.897 | 0.032 | 0.868 ± 0.050 |
| Decision Tree | 0.744 | 0.754 | 0.871 | 0.116 | 0.754 ± 0.098 |
| Random Forest | 0.814 | 0.803 | 1.000 | 0.197 | 0.811 ± 0.092 |

- Logistic Regression achieved the strongest generalisation performance, with a test F1 of 0.865 and a cross-validated F1 of 0.868 ± 0.050 presenting the tightest standard deviation of the three models, indicating stable and consistent performance across measures. 

- The Random Forest reached perfect train F1 (1.000), indicating memorisation of the training data, with a corresponding test F1 of 0.803. 

- The Decision Tree showed the largest instability, with a CV standard deviation of 0.098. Based on
these results, Logistic Regression was selected as the final model. Its ROC-AUC on the held-out test set was 0.942, indicating great discriminative ability between placed and not-placed candidates on genuinely unseen data.

## Bias awareness

Two sources of potential bias were identified. First, the dataset exhibits a gender placement gapin which female students were placed at 63.2% versus 71.9% for male students. The trained model reproduces this gap closely (predicted rates: 67.1% F, 72.7% M), suggesting it has learned the historical disparity rather than correcting for it. Second, degree stream shows a 24.8 percentage point placement gap between Comm&Mgmt (70.3%) and the Others category (45.5%). 

Since undergraduate subject choice correlates with socioeconomic background in many educational systems,`degree_t` may function as a class proxy, meaning the model could be encoding structural disadvantage rather than academic merit. Feature importance analysis confirmed that the top predictors were all continuous academic scores (`ssc_p`, `degree_p`, `hsc_p`), with demographic features ranking lower.

However, their presence in the feature matrix is sufficient to transmit group-level disparities into predictions. Responsible deployment of this model would require subgroup fairness auditing, threshold calibration per group, or removal of demographic features alongside monitoring for proxy transmission through correlated scores.